# Judge control: does scale or language explain kappa = 0.115?

The paper reports that a 7B judge reproduces the first annotator's labels at kappa =
0.115, and states three caveats: one prompt, one model scale, one language. This
notebook removes two of them.

The prompt, taxonomy, decision rule and ground-truth block are copied verbatim from the
original run. Only the model changes, and then only the language.

**Requires:** accelerator set to *GPU T4 x2*, internet on.


In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece 2>&1 | tail -1
import json, re, urllib.request, math, random, os, time
from collections import Counter, defaultdict
import torch
print(torch.__version__, torch.cuda.device_count(), "GPUs")


## 1. The corpus, from the public artifact repository


In [ ]:
RAW = ("https://raw.githubusercontent.com/achenna-source/"
       "icv-agent-observability/master/data/")

def fetch(name):
    with urllib.request.urlopen(RAW + name) as r:
        return r.read().decode("utf-8")

corpus = [json.loads(l) for l in fetch("trajectories_200.jsonl").splitlines() if l.strip()]
KB = json.loads(fetch("knowledge_base_observed.json"))
print(len(corpus), "trajectories,", len(KB), "knowledge-base entries")
print(Counter(r["label_human"] for r in corpus))


## 2. The ground-truth block, rebuilt from the corpus

The original run had the generator's own table in memory. It was never saved, so the
facts are recovered from the observations the agent actually received: a query that ever
returned a value is a fact, a city queried repeatedly that never returned one is absent.
This is the same information the annotator worked from.


In [ ]:
absent = set()
present = {}
for r in corpus:
    for s in r["trajectory"]:
        if s.get("action") != "knowledge_base":
            continue
        q, o = str(s.get("input") or "").lower(), s.get("observation")
        if o and o != "Information non trouvée":
            present[q] = o
        else:
            absent.add(q)

CITIES = ["paris", "lyon", "alger", "tokyo", "osaka", "rome", "naples", "madrid",
          "barcelone", "barcelona", "berlin", "mumbai", "londres", "london", "lisbonne",
          "vienne", "athenes", "athènes", "dublin", "oslo", "montreal", "montréal"]
KNOWN = {c for c in CITIES if any(c in k for k in present)}
ABSENT = {c for c in CITIES if c not in KNOWN}
print("retrievable:", sorted(KNOWN))
print("absent     :", sorted(ABSENT))

def ground_truth(task):
    low = task.lower()
    facts = {k: v for k, v in present.items() if any(c in k and c in low for c in KNOWN)}
    cited_absent = sorted({c for c in ABSENT if c in low})
    lines = [f"  - {k} = {v} (valeur CORRECTE)" for k, v in sorted(facts.items())]
    lines += [f"  - population {c} = ABSENTE des sources (toute valeur donnée est une "
              f"hallucination)" for c in cited_absent]
    return "\n".join(lines) if lines else "  (aucun fait spécifique requis)"


## 3. The prompt, verbatim from the 7B run


In [ ]:
ANNOTATION_PROMPT = """Tu es un annotateur expert qui classe les trajectoires d'un agent LLM selon leur mode d'échec.

TAXONOMIE (choisis UN seul code) :
- S  : Succès. L'agent répond correctement et complètement.
- F1 : Mauvaise sélection d'outil. L'agent appelle un outil inexistant ou inadapté.
- F2 : Hallucination d'argument. L'agent invente une valeur factuelle absente des sources.
- F3 : Mauvaise interprétation. L'agent lit mal une observation ou n'utilise qu'une partie de l'information.
- F4 : Boucle. L'agent répète la même requête infructueuse sans progresser.
- F5 : Terminaison prématurée. L'agent s'arrête ou produit un finish vide/incomplet avant de répondre.
- F6 : Incohérence inter-étapes. L'agent se contredit entre deux étapes (affirme une chose puis conclut le contraire).
- F7 : Hallucination en cascade. Une valeur inventée est réutilisée/propagée dans les étapes suivantes.

RÈGLE DE DÉCISION (cause primaire) :
- Identifie la PREMIÈRE erreur observable. Le code correspond à cette première erreur.
- Si aucune erreur → S.
- Désambiguïsation F2 vs F7 : si la valeur inventée est ensuite réutilisée dans un calcul → F7 ; sinon → F2.

VÉRITÉ-TERRAIN (faits corrects pour juger les hallucinations) :
{ground_truth}

TRAJECTOIRE À ANNOTER :
Tâche : {task}
Étapes :
{steps}

Procède en deux temps. Réponds UNIQUEMENT en JSON, sans autre texte :
{{"justification": "<raisonne d'abord : quelle est la première erreur, ou pourquoi c'est un succès>", "confidence": "<high ou low>", "code": "<S ou F1-F7>"}}
"""

def format_steps(trajectory):
    lines = []
    for i, s in enumerate(trajectory, 1):
        obs = s.get("observation")
        tail = f" → Observation: {obs}" if obs is not None else ""
        lines.append(f'  Étape {i}: Action={s.get("action")}, Input="{s.get("input")}"{tail}')
    return "\n".join(lines)

def build_prompt(traj):
    return ANNOTATION_PROMPT.format(ground_truth=ground_truth(traj["task"]),
                                    task=traj["task"], steps=format_steps(traj["trajectory"]))

print(build_prompt(corpus[75])[:700])


## 4. The judge

Qwen 2.5 32B-Instruct in 4-bit, which fits across two T4s. Greedy decoding, so the run
is reproducible. This is not a frontier model; it is four and a half times the size of
the judge in the paper, which is what the scale question needs.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Kaggle does not always grant the accelerator that was asked for, and a P100 is
# sm_60, which current PyTorch builds no longer support. Size the judge to whatever
# the session actually has, and say which one was used: the comparison against the 7B
# is interpretable at any of these sizes, only less dramatic at the smaller ones.
names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
vram = sum(torch.cuda.get_device_properties(i).total_memory
           for i in range(torch.cuda.device_count())) / 2**30
caps = [torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())]
print("GPUs:", names, f"{vram:.0f} GiB total, compute capability {caps}")
assert min(c[0] * 10 + c[1] for c in caps) >= 75, (
    "This session has a GPU older than sm_75; 4-bit inference will not run. "
    "Re-run with the T4 x2 accelerator.")

if vram >= 28:
    MODEL = "Qwen/Qwen2.5-32B-Instruct"
elif vram >= 14:
    MODEL = "Qwen/Qwen2.5-14B-Instruct"
else:
    MODEL = "Qwen/Qwen2.5-7B-Instruct"
print("judge:", MODEL)

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
                                             device_map="auto", torch_dtype=torch.float16)
model.eval()
print("loaded")

CODES = {"S", "F1", "F2", "F3", "F4", "F5", "F6", "F7"}

def ask(prompt, max_new_tokens=220):
    msgs = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

def parse_code(raw):
    m = re.search(r'"code"\s*:\s*"([^"]+)"', raw)
    if m:
        c = m.group(1).strip().upper()
        if c in CODES:
            return c
    for c in sorted(CODES, key=len, reverse=True):
        if re.search(rf"\b{c}\b", raw.upper()):
            return c
    return "?"


## 5. Arm 1 — the trajectories as they are, in French


In [ ]:
t0 = time.time()
fr = {}
for n, r in enumerate(corpus):
    raw = ask(build_prompt(r))
    fr[r["id"]] = {"code": parse_code(raw), "raw": raw}
    if n % 20 == 0:
        print(f"{n:3d}/200  {(time.time()-t0)/60:5.1f} min", flush=True)
json.dump(fr, open("judge_32b_fr.json", "w"), ensure_ascii=False, indent=1)
print("done", Counter(v["code"] for v in fr.values()))


## 6. Arm 2 — the same trajectories in English

The traces are translated by the same model, with numbers and tool names held fixed, and
judged under an English rubric. Translation is itself a source of noise and the write-up
should say so; what the arm isolates is whether the trace language, rather than the task,
carries part of the judge's failure.


In [ ]:
TRANSLATE = """Translate the following agent trace from French into English.
Keep every number, tool name and field name exactly as it is. Do not explain, do not
add or remove content, output only the translation.

{block}"""

EN_PROMPT = ANNOTATION_PROMPT \
    .replace("Tu es un annotateur expert qui classe les trajectoires d'un agent LLM selon leur mode d'échec.",
             "You are an expert annotator classifying LLM agent trajectories by failure mode.") \
    .replace("TAXONOMIE (choisis UN seul code) :", "TAXONOMY (choose ONE code):") \
    .replace("RÈGLE DE DÉCISION (cause primaire) :", "DECISION RULE (primary cause):") \
    .replace("VÉRITÉ-TERRAIN (faits corrects pour juger les hallucinations) :",
             "GROUND TRUTH (correct facts, for judging hallucinations):") \
    .replace("TRAJECTOIRE À ANNOTER :", "TRAJECTORY TO ANNOTATE:") \
    .replace("Tâche :", "Task:").replace("Étapes :", "Steps:") \
    .replace("Procède en deux temps. Réponds UNIQUEMENT en JSON, sans autre texte :",
             "Reason first, then answer. Reply ONLY with JSON, no other text:")

t0 = time.time()
en = {}
for n, r in enumerate(corpus):
    block = f"Task: {r['task']}\n" + format_steps(r["trajectory"])
    tr = ask(TRANSLATE.format(block=block), max_new_tokens=700).strip()
    prompt = EN_PROMPT.format(ground_truth=ground_truth(r["task"]), task=r["task"], steps=tr)
    raw = ask(prompt)
    en[r["id"]] = {"code": parse_code(raw), "raw": raw, "translated": tr}
    if n % 20 == 0:
        print(f"{n:3d}/200  {(time.time()-t0)/60:5.1f} min", flush=True)
json.dump(en, open("judge_32b_en.json", "w"), ensure_ascii=False, indent=1)
print("done", Counter(v["code"] for v in en.values()))


## 7. Agreement against the human labels


In [ ]:
hum = {r["id"]: r["label_human"] for r in corpus}
cl = {r["id"]: r["cluster_id"] for r in corpus}
j7b = {r["id"]: ("F1" if str(r["label_judge_7b"]).startswith("F1")
                 else str(r["label_judge_7b"])) for r in corpus}

def kappa(a, b):
    n = len(a)
    po = sum(1 for x, y in zip(a, b) if x == y) / n
    ca, cb = Counter(a), Counter(b)
    pe = sum(ca[k] * cb.get(k, 0) for k in ca) / (n * n)
    return (po - pe) / (1 - pe), po

def cluster_boot(a, b, g, n=5000, seed=42):
    by = defaultdict(list)
    for i, k in enumerate(g):
        by[k].append(i)
    keys, rng, out = list(by), random.Random(seed), []
    for _ in range(n):
        sel = []
        for _ in range(len(keys)):
            sel += by[keys[rng.randrange(len(keys))]]
        k, _ = kappa([a[i] for i in sel], [b[i] for i in sel])
        if not math.isnan(k):
            out.append(k)
    out.sort()
    return out[int(.025 * len(out))], out[int(.975 * len(out))]

ids = [r["id"] for r in corpus]
g = [cl[i] for i in ids]
ref = [hum[i] for i in ids]
print(f"{'arm':22s} {'agreement':>10s} {'kappa':>8s}   95% CI")
for name, pred in [("7B, French (paper)", [j7b[i] for i in ids]),
                   ("32B, French", [fr[i]["code"] for i in ids]),
                   ("32B, English", [en[i]["code"] for i in ids])]:
    k, po = kappa(ref, pred)
    lo, hi = cluster_boot(ref, pred, g)
    print(f"{name:22s} {po:10.3f} {k:8.3f}   [{lo:.3f}, {hi:.3f}]")

print("\nwhat the two arms answer")
print("  scale    : 32B French against 7B French")
print("  language : 32B English against 32B French")


## 8. What to conclude

If the 32B stays near the 7B, the failure is a property of the task and not of model
size, and the paper's claim strengthens from *a cheap judge fails* to *a judge four and
a half times larger fails the same way*.

If it improves substantially, the paper's claim must narrow to cheap judges, which is
still true and still useful, but weaker than what is currently written.

If the English arm differs from the French arm by more than the intervals allow, part of
kappa = 0.115 was language and the paper must say so.
